In [44]:
# Import libraries
import pandas as pd
import numpy as np

In [45]:
# Load data
path = "D:/D/Projects/Healthcare Analytics/data/raw"

admissions = pd.read_csv(path+'/admissions.csv')
claims = pd.read_csv(path+'/claims.csv')
patients = pd.read_csv(path+'/patients.csv')
readmissions = pd.read_csv(path+'/readmissions.csv')

CREATE DATA QUALITY FLAGS

In [46]:
# Convert data column firt, try to do it first
admissions['admit_date'] = pd.to_datetime(admissions['admit_date'])
admissions['discharge_date'] = pd.to_datetime(admissions['discharge_date'])

In [47]:
# Start generate flag for los, dignosis, billing, claims
# Flag LOS data, and count negative day stay because it is impossible dtays
admissions['negative_los_flag'] = admissions['los_days']<0
print('Negative LOS count: ', (admissions['los_days']<0).sum())

# Date Logic error (litle different than negative los)
admissions['date_logic_error_flag'] = (admissions['discharge_date'] < admissions['admit_date'])
print(admissions['date_logic_error_flag'].sum())

# Try to check LOS mismatch
computed_los = (
    admissions['discharge_date'] - admissions['admit_date']
).dt.days

admissions['los_mismatch_flag'] = (
    admissions['los_days'] != computed_los
)
print("LOS mismatch count: ", admissions['los_mismatch_flag'].sum())

Negative LOS count:  2000
2000
LOS mismatch count:  3974


In [48]:
# missing diagnosis codes with ICD format check
admissions['diagnosis_missing_flag'] = (
    admissions['diagnosis_code'].isna()
)

# Perform standardization on diagnosis data to clea it. 
icd_pattern = r"^[A-Z][0-9]{2}(\.[A-Z0-9]{1,3})?$"

admissions['diagnosis_invalid_format_flag'] = (
    admissions['diagnosis_code']
    .astype("string")
    .str.strip()
    .str.upper()
    .str.match(icd_pattern, na=False)
) == False

# Suspicious diagnosis description
admissions['diagnosis_desc_invalid_flag'] = (
    admissions['diagnosis_desc']
    .astype("string")
    .str.strip()
    .str.upper()
    .isin(["INVALID", "N/A", "MISSING", ""])
)

In [50]:
# Orphan patient records
admissions['orphan_patient_flag'] = (
    ~admissions['patient_id'].isin(patients['patient_id'])
)

print("Orphan admissions count: ", admissions['orphan_patient_flag'].sum())

Orphan admissions count:  1000
